Importação das bibliotecas necessárias.
- `pandas as pd` para leitura e tratativa do dataset
- `numpy as np` para cálculos e processamentos de dados do dataset
- o `sklearn` é muito usado para machine learning, nesse caso foram importadas as bibliotecas `MinMaxScaler` e `StandardScaler`, elas serão responsáveis para a tratativa e padronização do dataframe.

In [41]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option('display.max_columns', None)

Fazendo a importação dos datasets e fazendo a tratativa de erros, caso o dataset tratado não exista, o algoritmo vai optar por carregar o dataset bruto.
> Primeiro, no `try`, o script vai tentar carregar os dados tratados, caso ele não encontre, o `except` vai fazer com que sejam carregados os dados brutos.
- Por fim, são exibidas as primeiras 5 linhas do dataset, a fim de verificar se foi importado corretamente.

In [42]:
arquivo_bruto = 'dados_aula1_machine_learning_industria.csv'
arquivo_tratado = 'dados_aula1_industriais_tratados.csv'

try:
  df = pd.read_csv(arquivo_tratado)
  print(f"Base carregada: {arquivo_tratado}")
except:
  df = pd.read_csv(arquivo_bruto)
  print(f"Base de dados carregada para fins didáticos!")

df.head()

Base carregada: dados_aula1_industriais_tratados.csv


,timestamp,lote_producao,operador,modo_controle,status_valvula,setpoint_vazao_l_min,vazao_medida_l_min,pressao_linha_bar,temperatura_processo_c,abertura_valvula_pct,vibracao_bomba_mm_s,tempo_resposta_ms,ph_produto,densidade_produto_g_cm3,status_instrumentacao
0,2026-01-10 08:00:00,Lote_A,op_b,automatico,normal,121.44,128.080,3.518,55.55625,56.97,2.960,220.8,7.09,1.069,estavel
1,2026-01-10 09:00:00,Lote_D,op_c,cascata,normal,126.17,119.320,4.616,43.34000,63.26,2.532,402.1,6.99,1.053,inspecao
2,2026-01-10 10:00:00,Lote_B,op_d,manual,intermitente,128.54,129.110,4.118,40.67000,63.14,2.182,284.1,6.72,1.021,inspecao
3,2026-01-10 11:00:00,Lote_C,op_a,automatico,intermitente,106.50,102.680,3.547,43.94000,64.75,2.772,486.3,6.67,1.093,inspecao
4,2026-01-10 12:00:00,Lote_C,op_a,automatico,normal,101.59,118.565,4.085,55.55625,83.82,3.173,283.3,6.66,1.004,estavel


Exibição de algumas informações sobre o dataframe, dentre elas:
- Dimensão do dataframe (Linhas x Colunas)
- Tipos de dados disponívis
- Colunas do dataframe

In [43]:
print(f"Dimensão da base: {df.shape}")

#Exibição dos tipos de dados do dataframe
display(df.dtypes)

#Exibição das colunas do dataframe
display(pd.DataFrame({'coluna': df.columns}))

Dimensão da base: (180, 15)


,0
timestamp,object
lote_producao,object
operador,object
modo_controle,object
status_valvula,object
setpoint_vazao_l_min,float64
vazao_medida_l_min,float64
pressao_linha_bar,float64
temperatura_processo_c,float64
abertura_valvula_pct,float64


,coluna
0,timestamp
1,lote_producao
2,operador
3,modo_controle
4,status_valvula
5,setpoint_vazao_l_min
6,vazao_medida_l_min
7,pressao_linha_bar
8,temperatura_processo_c
9,abertura_valvula_pct


Identificação das colunas numéricas e categóricas
- Essa separação é feita através da função `select_dtypes()`, nela é incluso tipos de dados que sejam separados em uma lista dentro do `include = []`
- Após essa separação, é feita a formatação para para lista com a função `.columns.tolist()`
- É importante que seja feita essa separação para poder identificar o eixo `Features`(`X`) e o `Alvo`(`y`)
> Exemplo de como é executado:<br>
  ```python
  alguns_dados = df.select_dtypes(include = ['Dados impactantes, cada tipo separado por vírgula']).columns.tolist()
  ```

In [44]:
colunas_numericas = df.select_dtypes(include = ['int64', 'float64']).columns.tolist()
colunas_categoricas = df.select_dtypes(include = ['object', 'category']).columns.tolist()

print('Colunas numéricas: ', colunas_numericas)
print('Colunas categóricas: ', colunas_categoricas)

Colunas numéricas:  ['setpoint_vazao_l_min', 'vazao_medida_l_min', 'pressao_linha_bar', 'temperatura_processo_c', 'abertura_valvula_pct', 'vibracao_bomba_mm_s', 'tempo_resposta_ms', 'ph_produto', 'densidade_produto_g_cm3']
Colunas categóricas:  ['timestamp', 'lote_producao', 'operador', 'modo_controle', 'status_valvula', 'status_instrumentacao']


Para esse exemplo, o eixo `y` será `status_instrumentacao`, portanto, é importante que essa coluna seja eliminada do DataFrame usado no eixo `X` e no eixo `y` seja atribuída apenas ela.
- No eixo `X` essa coluna é eliminada através da função `.drop()`, dentro do parênteses é indicado quais as colunas desejadas pelo comando `columns = []` e, por fim, é usado a função `.copy()` para que essa alteração não seja aplicada no DataFrame original.
> Exemplo:
  ```python
  X = df.drop(columns = ['Insira_coluna_desejada']).copy()
  ```

In [45]:
alvo = 'status_instrumentacao'

X = df.drop(columns = [alvo]).copy()
y = df[alvo].copy()

print('Dimensão do X: ', X.shape)
print('Dimensão do y: ', y.shape)

display(y.value_counts(dropna = False).rename_axis('classe').reset_index(name = 'quantidade'))

Dimensão do X:  (180, 14)
Dimensão do y:  (180,)


,classe,quantidade
0,inspecao,116
1,estavel,64


Dentro dos dados na variável `X`, é feita a separação das colunas numéricas e categóricas, novamente, usando a função `.select_dtypes()`

In [46]:
X_numerico = X.select_dtypes(include = ['int64', 'float64']).copy()
X_categorico = X.select_dtypes(include = ['object', 'category']).copy()

display(X_numerico.head())
display(X_categorico.head())

,setpoint_vazao_l_min,vazao_medida_l_min,pressao_linha_bar,temperatura_processo_c,abertura_valvula_pct,vibracao_bomba_mm_s,tempo_resposta_ms,ph_produto,densidade_produto_g_cm3
0,121.44,128.080,3.518,55.55625,56.97,2.960,220.8,7.09,1.069
1,126.17,119.320,4.616,43.34000,63.26,2.532,402.1,6.99,1.053
2,128.54,129.110,4.118,40.67000,63.14,2.182,284.1,6.72,1.021
3,106.50,102.680,3.547,43.94000,64.75,2.772,486.3,6.67,1.093
4,101.59,118.565,4.085,55.55625,83.82,3.173,283.3,6.66,1.004


,timestamp,lote_producao,operador,modo_controle,status_valvula
0,2026-01-10 08:00:00,Lote_A,op_b,automatico,normal
1,2026-01-10 09:00:00,Lote_D,op_c,cascata,normal
2,2026-01-10 10:00:00,Lote_B,op_d,manual,intermitente
3,2026-01-10 11:00:00,Lote_C,op_a,automatico,intermitente
4,2026-01-10 12:00:00,Lote_C,op_a,automatico,normal


**Normalização dos dados numéricos presentes no DataFrame.**<br>
A normalização com `MinMaxScaler()` é muito usada quando queremos que os dados numéricos tenham valores limitados (Todos os dados numéricos vão de 0 à 1).

In [47]:
scaler_minmax = MinMaxScaler()

X_normalizado_array = scaler_minmax.fit_transform(X_numerico)

X_normalizado = pd.DataFrame(X_normalizado_array, columns = X_numerico.columns, index = X_numerico.index)

X_normalizado.head()

,setpoint_vazao_l_min,vazao_medida_l_min,pressao_linha_bar,temperatura_processo_c,abertura_valvula_pct,vibracao_bomba_mm_s,tempo_resposta_ms,ph_produto,densidade_produto_g_cm3
0,0.466897,0.578050,0.335708,1.000000,0.479390,0.691935,0.197053,0.690789,0.648649
1,0.537907,0.444266,0.857075,0.507211,0.556786,0.576882,0.708333,0.625000,0.562162
2,0.573487,0.593780,0.620608,0.399506,0.555309,0.482796,0.375564,0.447368,0.389189
3,0.242606,0.190138,0.349478,0.531414,0.575120,0.641398,0.945784,0.414474,0.778378
4,0.168894,0.432736,0.604938,1.000000,0.809770,0.749194,0.373308,0.407895,0.297297


> Exemplo dos dados não normalizados com os normalizados.

In [48]:
comparacao_normalizacao = pd.DataFrame({
    'valor_original_min': X_numerico.min(),
    'valor_original_max': X_numerico.max(),
    'valor_normalizado_min': X_normalizado.min(),
    'valor_normalizado_max': X_normalizado.max()
})

display(comparacao_normalizacao)

,valor_original_min,valor_original_max,valor_normalizado_min,valor_normalizado_max
setpoint_vazao_l_min,90.34000,156.95000,0.0,1.0
vazao_medida_l_min,90.23000,155.70875,0.0,1.0
pressao_linha_bar,2.81100,4.91700,0.0,1.0
temperatura_processo_c,30.76625,55.55625,0.0,1.0
abertura_valvula_pct,18.01000,99.28000,0.0,1.0
vibracao_bomba_mm_s,0.38600,4.10600,0.0,1.0
tempo_resposta_ms,150.92500,505.52500,0.0,1.0
ph_produto,6.04000,7.56000,0.0,1.0
densidade_produto_g_cm3,0.94900,1.13400,0.0,1.0


**Padronização dos dados.**<br>
Normaliza os dados em faixas numéricas, muito usada quando esses dados tem mais foco em média ou variância unitária.

In [49]:
scaler_standard = StandardScaler()

X_padronizado_array = scaler_standard.fit_transform(X_numerico)

X_padronizado = pd.DataFrame(X_padronizado_array, columns = X_numerico.columns, index = X_numerico.index)

X_padronizado.head()

,setpoint_vazao_l_min,vazao_medida_l_min,pressao_linha_bar,temperatura_processo_c,abertura_valvula_pct,vibracao_bomba_mm_s,tempo_resposta_ms,ph_produto,densidade_produto_g_cm3
0,0.191198,0.639145,-0.856294,2.317030,-0.096176,0.971771,-1.574939,0.916810,0.922717
1,0.577176,-0.021791,1.750977,0.040508,0.290154,0.368680,1.087681,0.595436,0.400630
2,0.770573,0.716858,0.568444,-0.457051,0.282783,-0.124501,-0.645298,-0.272276,-0.643545
3,-1.027936,-1.277268,-0.787432,0.152319,0.381669,0.706862,2.324265,-0.432963,1.705849
4,-1.428602,-0.078755,0.490084,2.317030,1.552943,1.271907,-0.657047,-0.465100,-1.198264


Comparação entre os dados não padronizados com os padronizados.

In [50]:
comparacao_padronizacao = pd.DataFrame({
    'media_original': X_numerico.mean(),
    'desvio_original': X_numerico.std(),
    'media_padronizada': X_padronizado.mean(),
    'desvio_padronizado': X_padronizado.std()
})

display(comparacao_padronizacao.round(4))

,media_original,desvio_original,media_padronizada,desvio_padronizado
setpoint_vazao_l_min,119.0969,12.2888,-0.0,1.0028
vazao_medida_l_min,119.6088,13.2909,0.0,1.0028
pressao_linha_bar,3.8786,0.4223,0.0,1.0028
temperatura_processo_c,43.1226,5.3812,0.0,1.0028
abertura_valvula_pct,58.5359,16.3268,0.0,1.0028
vibracao_bomba_mm_s,2.2704,0.7117,-0.0,1.0028
tempo_resposta_ms,328.0389,68.2808,-0.0,1.0028
ph_produto,6.8047,0.3120,-0.0,1.0028
densidade_produto_g_cm3,1.0407,0.0307,0.0,1.0028


**Codificação de variáveis categóricas**<br>
Transformar variáveis categóricas em colunas numéricas com a função `pd.get_dummies()`. Cada categoria vira uma coluna, sendo `1` indicando que aquela linha possui a categoria e `0` indicando que não possui.

- Geralmente usado quando os modelos precisam de uma entrada numérica (como regressão linear, árvore de decisão ou redes neurais).
- Pode ser usado também quando as categorias não possuem uma ordem natural (tipo as cores Vermelho, Amarelo e Azul, elas não recebem valores como 1, 2 e 3), então, é gerada uma hierarquia artificial.

In [51]:
X_categorico_codificado = pd.get_dummies(X_categorico, drop_first = False)

print('Dimensão da base categórica codificada: ', X_categorico_codificado.shape)
X_categorico_codificado.head()

Dimensão da base categórica codificada:  (180, 194)


,timestamp_2026-01-10 08:00:00,timestamp_2026-01-10 09:00:00,timestamp_2026-01-10 10:00:00,timestamp_2026-01-10 11:00:00,timestamp_2026-01-10 12:00:00,timestamp_2026-01-10 13:00:00,timestamp_2026-01-10 14:00:00,timestamp_2026-01-10 15:00:00,timestamp_2026-01-10 16:00:00,timestamp_2026-01-10 17:00:00,timestamp_2026-01-10 18:00:00,timestamp_2026-01-10 19:00:00,timestamp_2026-01-10 20:00:00,timestamp_2026-01-10 21:00:00,timestamp_2026-01-10 22:00:00,timestamp_2026-01-10 23:00:00,timestamp_2026-01-11 00:00:00,timestamp_2026-01-11 01:00:00,timestamp_2026-01-11 02:00:00,timestamp_2026-01-11 03:00:00,timestamp_2026-01-11 04:00:00,timestamp_2026-01-11 05:00:00,timestamp_2026-01-11 06:00:00,timestamp_2026-01-11 07:00:00,timestamp_2026-01-11 08:00:00,timestamp_2026-01-11 09:00:00,timestamp_2026-01-11 10:00:00,timestamp_2026-01-11 11:00:00,timestamp_2026-01-11 12:00:00,timestamp_2026-01-11 13:00:00,timestamp_2026-01-11 14:00:00,timestamp_2026-01-11 15:00:00,timestamp_2026-01-11 16:00:00,timestamp_2026-01-11 17:00:00,timestamp_2026-01-11 18:00:00,timestamp_2026-01-11 19:00:00,timestamp_2026-01-11 20:00:00,timestamp_2026-01-11 21:00:00,timestamp_2026-01-11 22:00:00,timestamp_2026-01-11 23:00:00,timestamp_2026-01-12 00:00:00,timestamp_2026-01-12 01:00:00,timestamp_2026-01-12 02:00:00,timestamp_2026-01-12 03:00:00,timestamp_2026-01-12 04:00:00,timestamp_2026-01-12 05:00:00,timestamp_2026-01-12 06:00:00,timestamp_2026-01-12 07:00:00,timestamp_2026-01-12 08:00:00,timestamp_2026-01-12 09:00:00,timestamp_2026-01-12 10:00:00,timestamp_2026-01-12 11:00:00,timestamp_2026-01-12 12:00:00,timestamp_2026-01-12 13:00:00,timestamp_2026-01-12 14:00:00,timestamp_2026-01-12 15:00:00,timestamp_2026-01-12 16:00:00,timestamp_2026-01-12 17:00:00,timestamp_2026-01-12 18:00:00,timestamp_2026-01-12 19:00:00,timestamp_2026-01-12 20:00:00,timestamp_2026-01-12 21:00:00,timestamp_2026-01-12 22:00:00,timestamp_2026-01-12 23:00:00,timestamp_2026-01-13 00:00:00,timestamp_2026-01-13 01:00:00,timestamp_2026-01-13 02:00:00,timestamp_2026-01-13 03:00:00,timestamp_2026-01-13 04:00:00,timestamp_2026-01-13 05:00:00,timestamp_2026-01-13 06:00:00,timestamp_2026-01-13 07:00:00,timestamp_2026-01-13 08:00:00,timestamp_2026-01-13 09:00:00,timestamp_2026-01-13 10:00:00,timestamp_2026-01-13 11:00:00,timestamp_2026-01-13 12:00:00,timestamp_2026-01-13 13:00:00,timestamp_2026-01-13 14:00:00,timestamp_2026-01-13 15:00:00,timestamp_2026-01-13 16:00:00,timestamp_2026-01-13 17:00:00,timestamp_2026-01-13 18:00:00,timestamp_2026-01-13 19:00:00,timestamp_2026-01-13 20:00:00,timestamp_2026-01-13 21:00:00,timestamp_2026-01-13 22:00:00,timestamp_2026-01-13 23:00:00,timestamp_2026-01-14 00:00:00,timestamp_2026-01-14 01:00:00,timestamp_2026-01-14 02:00:00,timestamp_2026-01-14 03:00:00,timestamp_2026-01-14 04:00:00,timestamp_2026-01-14 05:00:00,timestamp_2026-01-14 06:00:00,timestamp_2026-01-14 07:00:00,timestamp_2026-01-14 08:00:00,timestamp_2026-01-14 09:00:00,timestamp_2026-01-14 10:00:00,timestamp_2026-01-14 11:00:00,timestamp_2026-01-14 12:00:00,timestamp_2026-01-14 13:00:00,timestamp_2026-01-14 14:00:00,timestamp_2026-01-14 15:00:00,timestamp_2026-01-14 16:00:00,timestamp_2026-01-14 17:00:00,timestamp_2026-01-14 18:00:00,timestamp_2026-01-14 19:00:00,timestamp_2026-01-14 20:00:00,timestamp_2026-01-14 21:00:00,timestamp_2026-01-14 22:00:00,timestamp_2026-01-14 23:00:00,timestamp_2026-01-15 00:00:00,timestamp_2026-01-15 01:00:00,timestamp_2026-01-15 02:00:00,timestamp_2026-01-15 03:00:00,timestamp_2026-01-15 04:00:00,timestamp_2026-01-15 05:00:00,timestamp_2026-01-15 06:00:00,timestamp_2026-01-15 07:00:00,timestamp_2026-01-15 08:00:00,timestamp_2026-01-15 09:00:00,timestamp_2026-01-15 10:00:00,timestamp_2026-01-15 11:00:00,timestamp_2026-01-15 12:00:00,timestamp_2026-01-15 13:00:00,timestamp_2026-01-15 14:00:00,timestamp_2026-01-15 15:00:00,timestamp_2026-01-15 16:00:00,timestamp_2026-01-15 17:00:00,timestamp_2026-01-15 18:00:00,timestamp_2026-01-15 19:00:00,timestamp_2026-01-15 20:00:00,timestamp

**Preparação da base final para modelagem**<br>
A base final possui:
- Variáveis numéricas padronizadas;
- Variáveis categóricas codificadas.

In [52]:
X_final = pd.concat([X_padronizado, X_categorico_codificado], axis = 1)

print('Dimensão da base final de atributos: ', X_final.shape)
display(X_final.head())

Dimensão da base final de atributos:  (180, 203)


,setpoint_vazao_l_min,vazao_medida_l_min,pressao_linha_bar,temperatura_processo_c,abertura_valvula_pct,vibracao_bomba_mm_s,tempo_resposta_ms,ph_produto,densidade_produto_g_cm3,timestamp_2026-01-10 08:00:00,timestamp_2026-01-10 09:00:00,timestamp_2026-01-10 10:00:00,timestamp_2026-01-10 11:00:00,timestamp_2026-01-10 12:00:00,timestamp_2026-01-10 13:00:00,timestamp_2026-01-10 14:00:00,timestamp_2026-01-10 15:00:00,timestamp_2026-01-10 16:00:00,timestamp_2026-01-10 17:00:00,timestamp_2026-01-10 18:00:00,timestamp_2026-01-10 19:00:00,timestamp_2026-01-10 20:00:00,timestamp_2026-01-10 21:00:00,timestamp_2026-01-10 22:00:00,timestamp_2026-01-10 23:00:00,timestamp_2026-01-11 00:00:00,timestamp_2026-01-11 01:00:00,timestamp_2026-01-11 02:00:00,timestamp_2026-01-11 03:00:00,timestamp_2026-01-11 04:00:00,timestamp_2026-01-11 05:00:00,timestamp_2026-01-11 06:00:00,timestamp_2026-01-11 07:00:00,timestamp_2026-01-11 08:00:00,timestamp_2026-01-11 09:00:00,timestamp_2026-01-11 10:00:00,timestamp_2026-01-11 11:00:00,timestamp_2026-01-11 12:00:00,timestamp_2026-01-11 13:00:00,timestamp_2026-01-11 14:00:00,timestamp_2026-01-11 15:00:00,timestamp_2026-01-11 16:00:00,timestamp_2026-01-11 17:00:00,timestamp_2026-01-11 18:00:00,timestamp_2026-01-11 19:00:00,timestamp_2026-01-11 20:00:00,timestamp_2026-01-11 21:00:00,timestamp_2026-01-11 22:00:00,timestamp_2026-01-11 23:00:00,timestamp_2026-01-12 00:00:00,timestamp_2026-01-12 01:00:00,timestamp_2026-01-12 02:00:00,timestamp_2026-01-12 03:00:00,timestamp_2026-01-12 04:00:00,timestamp_2026-01-12 05:00:00,timestamp_2026-01-12 06:00:00,timestamp_2026-01-12 07:00:00,timestamp_2026-01-12 08:00:00,timestamp_2026-01-12 09:00:00,timestamp_2026-01-12 10:00:00,timestamp_2026-01-12 11:00:00,timestamp_2026-01-12 12:00:00,timestamp_2026-01-12 13:00:00,timestamp_2026-01-12 14:00:00,timestamp_2026-01-12 15:00:00,timestamp_2026-01-12 16:00:00,timestamp_2026-01-12 17:00:00,timestamp_2026-01-12 18:00:00,timestamp_2026-01-12 19:00:00,timestamp_2026-01-12 20:00:00,timestamp_2026-01-12 21:00:00,timestamp_2026-01-12 22:00:00,timestamp_2026-01-12 23:00:00,timestamp_2026-01-13 00:00:00,timestamp_2026-01-13 01:00:00,timestamp_2026-01-13 02:00:00,timestamp_2026-01-13 03:00:00,timestamp_2026-01-13 04:00:00,timestamp_2026-01-13 05:00:00,timestamp_2026-01-13 06:00:00,timestamp_2026-01-13 07:00:00,timestamp_2026-01-13 08:00:00,timestamp_2026-01-13 09:00:00,timestamp_2026-01-13 10:00:00,timestamp_2026-01-13 11:00:00,timestamp_2026-01-13 12:00:00,timestamp_2026-01-13 13:00:00,timestamp_2026-01-13 14:00:00,timestamp_2026-01-13 15:00:00,timestamp_2026-01-13 16:00:00,timestamp_2026-01-13 17:00:00,timestamp_2026-01-13 18:00:00,timestamp_2026-01-13 19:00:00,timestamp_2026-01-13 20:00:00,timestamp_2026-01-13 21:00:00,timestamp_2026-01-13 22:00:00,timestamp_2026-01-13 23:00:00,timestamp_2026-01-14 00:00:00,timestamp_2026-01-14 01:00:00,timestamp_2026-01-14 02:00:00,timestamp_2026-01-14 03:00:00,timestamp_2026-01-14 04:00:00,timestamp_2026-01-14 05:00:00,timestamp_2026-01-14 06:00:00,timestamp_2026-01-14 07:00:00,timestamp_2026-01-14 08:00:00,timestamp_2026-01-14 09:00:00,timestamp_2026-01-14 10:00:00,timestamp_2026-01-14 11:00:00,timestamp_2026-01-14 12:00:00,timestamp_2026-01-14 13:00:00,timestamp_2026-01-14 14:00:00,timestamp_2026-01-14 15:00:00,timestamp_2026-01-14 16:00:00,timestamp_2026-01-14 17:00:00,timestamp_2026-01-14 18:00:00,timestamp_2026-01-14 19:00:00,timestamp_2026-01-14 20:00:00,timestamp_2026-01-14 21:00:00,timestamp_2026-01-14 22:00:00,timestamp_2026-01-14 23:00:00,timestamp_2026-01-15 00:00:00,timestamp_2026-01-15 01:00:00,timestamp_2026-01-15 02:00:00,timestamp_2026-01-15 03:00:00,timestamp_2026-01-15 04:00:00,timestamp_2026-01-15 05:00:00,timestamp_2026-01-15 06:00:00,timestamp_2026-01-15 07:00:00,timestamp_2026-01-15 08:00:00,timestamp_2026-01-15 09:00:00,timestamp_2026-01-15 10:00:00,timestamp_2026-01-15 11:00:00,timestamp_2026-01-15 12:00:00,timestamp_2026-01-15 13:00:00,timestamp_2026-01-15 14:00:00,timestamp_2026

In [53]:
base_final_modelagem = X_final.copy()
base_final_modelagem[alvo] = y.values

print("Dimensão da base final completa: ", base_final_modelagem.shape)
display(base_final_modelagem.head())

Dimensão da base final completa:  (180, 204)


,setpoint_vazao_l_min,vazao_medida_l_min,pressao_linha_bar,temperatura_processo_c,abertura_valvula_pct,vibracao_bomba_mm_s,tempo_resposta_ms,ph_produto,densidade_produto_g_cm3,timestamp_2026-01-10 08:00:00,timestamp_2026-01-10 09:00:00,timestamp_2026-01-10 10:00:00,timestamp_2026-01-10 11:00:00,timestamp_2026-01-10 12:00:00,timestamp_2026-01-10 13:00:00,timestamp_2026-01-10 14:00:00,timestamp_2026-01-10 15:00:00,timestamp_2026-01-10 16:00:00,timestamp_2026-01-10 17:00:00,timestamp_2026-01-10 18:00:00,timestamp_2026-01-10 19:00:00,timestamp_2026-01-10 20:00:00,timestamp_2026-01-10 21:00:00,timestamp_2026-01-10 22:00:00,timestamp_2026-01-10 23:00:00,timestamp_2026-01-11 00:00:00,timestamp_2026-01-11 01:00:00,timestamp_2026-01-11 02:00:00,timestamp_2026-01-11 03:00:00,timestamp_2026-01-11 04:00:00,timestamp_2026-01-11 05:00:00,timestamp_2026-01-11 06:00:00,timestamp_2026-01-11 07:00:00,timestamp_2026-01-11 08:00:00,timestamp_2026-01-11 09:00:00,timestamp_2026-01-11 10:00:00,timestamp_2026-01-11 11:00:00,timestamp_2026-01-11 12:00:00,timestamp_2026-01-11 13:00:00,timestamp_2026-01-11 14:00:00,timestamp_2026-01-11 15:00:00,timestamp_2026-01-11 16:00:00,timestamp_2026-01-11 17:00:00,timestamp_2026-01-11 18:00:00,timestamp_2026-01-11 19:00:00,timestamp_2026-01-11 20:00:00,timestamp_2026-01-11 21:00:00,timestamp_2026-01-11 22:00:00,timestamp_2026-01-11 23:00:00,timestamp_2026-01-12 00:00:00,timestamp_2026-01-12 01:00:00,timestamp_2026-01-12 02:00:00,timestamp_2026-01-12 03:00:00,timestamp_2026-01-12 04:00:00,timestamp_2026-01-12 05:00:00,timestamp_2026-01-12 06:00:00,timestamp_2026-01-12 07:00:00,timestamp_2026-01-12 08:00:00,timestamp_2026-01-12 09:00:00,timestamp_2026-01-12 10:00:00,timestamp_2026-01-12 11:00:00,timestamp_2026-01-12 12:00:00,timestamp_2026-01-12 13:00:00,timestamp_2026-01-12 14:00:00,timestamp_2026-01-12 15:00:00,timestamp_2026-01-12 16:00:00,timestamp_2026-01-12 17:00:00,timestamp_2026-01-12 18:00:00,timestamp_2026-01-12 19:00:00,timestamp_2026-01-12 20:00:00,timestamp_2026-01-12 21:00:00,timestamp_2026-01-12 22:00:00,timestamp_2026-01-12 23:00:00,timestamp_2026-01-13 00:00:00,timestamp_2026-01-13 01:00:00,timestamp_2026-01-13 02:00:00,timestamp_2026-01-13 03:00:00,timestamp_2026-01-13 04:00:00,timestamp_2026-01-13 05:00:00,timestamp_2026-01-13 06:00:00,timestamp_2026-01-13 07:00:00,timestamp_2026-01-13 08:00:00,timestamp_2026-01-13 09:00:00,timestamp_2026-01-13 10:00:00,timestamp_2026-01-13 11:00:00,timestamp_2026-01-13 12:00:00,timestamp_2026-01-13 13:00:00,timestamp_2026-01-13 14:00:00,timestamp_2026-01-13 15:00:00,timestamp_2026-01-13 16:00:00,timestamp_2026-01-13 17:00:00,timestamp_2026-01-13 18:00:00,timestamp_2026-01-13 19:00:00,timestamp_2026-01-13 20:00:00,timestamp_2026-01-13 21:00:00,timestamp_2026-01-13 22:00:00,timestamp_2026-01-13 23:00:00,timestamp_2026-01-14 00:00:00,timestamp_2026-01-14 01:00:00,timestamp_2026-01-14 02:00:00,timestamp_2026-01-14 03:00:00,timestamp_2026-01-14 04:00:00,timestamp_2026-01-14 05:00:00,timestamp_2026-01-14 06:00:00,timestamp_2026-01-14 07:00:00,timestamp_2026-01-14 08:00:00,timestamp_2026-01-14 09:00:00,timestamp_2026-01-14 10:00:00,timestamp_2026-01-14 11:00:00,timestamp_2026-01-14 12:00:00,timestamp_2026-01-14 13:00:00,timestamp_2026-01-14 14:00:00,timestamp_2026-01-14 15:00:00,timestamp_2026-01-14 16:00:00,timestamp_2026-01-14 17:00:00,timestamp_2026-01-14 18:00:00,timestamp_2026-01-14 19:00:00,timestamp_2026-01-14 20:00:00,timestamp_2026-01-14 21:00:00,timestamp_2026-01-14 22:00:00,timestamp_2026-01-14 23:00:00,timestamp_2026-01-15 00:00:00,timestamp_2026-01-15 01:00:00,timestamp_2026-01-15 02:00:00,timestamp_2026-01-15 03:00:00,timestamp_2026-01-15 04:00:00,timestamp_2026-01-15 05:00:00,timestamp_2026-01-15 06:00:00,timestamp_2026-01-15 07:00:00,timestamp_2026-01-15 08:00:00,timestamp_2026-01-15 09:00:00,timestamp_2026-01-15 10:00:00,timestamp_2026-01-15 11:00:00,timestamp_2026-01-15 12:00:00,timestamp_2026-01-15 13:00:00,timestamp_2026-01-15 14:00:00,timestamp_2026

In [56]:
display(base_final_modelagem.isnull().sum().rename_axis('coluna').reset_index(name = 'faltantes'))
display(base_final_modelagem.dtypes.rename_axis('coluna').reset_index(name = 'tipo'))

,coluna,faltantes
0,setpoint_vazao_l_min,0
1,vazao_medida_l_min,0
2,pressao_linha_bar,0
3,temperatura_processo_c,0
4,abertura_valvula_pct,0
...,...,...
199,modo_controle_manual,0
200,status_valvula_intermitente,0
201,status_valvula_normal,0
202,status_valvula_travada,0


,coluna,tipo
0,setpoint_vazao_l_min,float64
1,vazao_medida_l_min,float64
2,pressao_linha_bar,float64
3,temperatura_processo_c,float64
4,abertura_valvula_pct,float64
...,...,...
199,modo_controle_manual,bool
200,status_valvula_intermitente,bool
201,status_valvula_normal,bool
202,status_valvula_travada,bool


**Categorização<br>**
Processo de representar atributos que descrevem grupos, classes ou rótulos. Em ML é preciso converter essas categorias em números e a forma correta depende se a categoria é nominal ou ordinal.<br>

| Tipo | Tem ordem natural | Exemplos |
| :---: | :---: | :---: |
| Nominal | Não | Cor: Azul, Verde e Vermelho |
| Ordinal | Sim | Escolaridade: Fundamental < Médio < Superior <br> Prioridade: Baixa < Média < Alta |
| Binário | Apenas duas classes | Sim / Não<br>Aprovado / Reprovado<br>Ativo / Inativo |

- One-hot / Dummies

---
Exportando os arquivos

In [57]:
X_final.to_csv('atributos_final_modelagem.csv', index = False)
base_final_modelagem.to_csv('base_final_modelagem.csv', index = False)